# Importador histórico Compra Ágil: Procesos + Ofertas — agosto 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Compra_Agil_Procesos_Ofertas_Colab.ipynb)

**Solo clon** · **una lectura de los dos CSV** · **sin API** · **simulación inicialmente** · **respaldo previo**.

Los archivos originales contienen filas de detalle repetidas, no necesariamente ofertas individuales. Se consolida `Procesos` por `CodigoCotizacion`. Para `Ofertas` se usa identificador sintético estable (SHA-256) construido desde código, RUT proveedor, monto, detalle textual y estado de selección. **No es un ID oficial:** dos cotizaciones idénticas del mismo proveedor no pueden distinguirse con certeza y quedan contabilizadas como una; revisar los indicadores antes de autorizar.

Esta versión no importa Resultados ni OrdenesCompra, no modifica producción, no toca GitHub Actions. La carga de ambas tablas es independiente; una falla no compromete la otra. La ejecución puede tardar por el tamaño del ZIP (1,41 GB descomprimidos).

In [ ]:
!pip -q install gspread google-api-python-client google-auth
from google.colab import auth,files
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload,MediaIoBaseDownload
from decimal import Decimal,InvalidOperation,ROUND_HALF_UP
import gspread,sqlite3,csv,zipfile,hashlib,io,json,os,datetime,time
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive'])
gc=gspread.authorize(creds);drive=build('drive','v3',credentials=creds)
CLON='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
HEAD_P=['codigo','nombre','estado','convocatoria','fecha_publicacion','fecha_cierre','monto_disponible_clp','organismo_comprador','region','total_ofertas_recibidas','motivo_seleccion','url_detalle','fecha_sincronizacion','categoria','subcategoria','modalidad']
HEAD_O=['codigo','proveedor','monto_ofertado_clp','fecha_oferta','fuente','id_cotizacion','seleccionado']
book=gc.open_by_key(CLON);sp=book.worksheet('Procesos');so=book.worksheet('Ofertas')
assert sp.row_values(1)==HEAD_P and so.row_values(1)==HEAD_O
print('Destino validado: clon, Procesos + Ofertas')

## Procesamiento de una sola lectura

Sube `COT_2026-08.zip` cuando se solicite. Los dos CSV se recorren una sola vez y se conservan datos consolidados en SQLite local. Se guarda una copia recuperable del SQLite en Google Drive al terminar. **No se escribe ninguna celda durante esta fase.**

In [ ]:
subidos=files.upload()
assert len(subidos)==1
zip_name=next(iter(subidos))
assert zip_name.lower().endswith('.zip')
dbpath='/content/compra_agil_preparado.sqlite'
if os.path.exists(dbpath):os.remove(dbpath)
db=sqlite3.connect(dbpath)
db.execute('PRAGMA journal_mode=DELETE')
db.execute('CREATE TABLE procesos (clave TEXT PRIMARY KEY, fila TEXT NOT NULL, calidad INT NOT NULL)')
db.execute('CREATE TABLE ofertas (clave TEXT PRIMARY KEY, fila TEXT NOT NULL)')
db.commit()
def limpio(s):return str(s or '').strip()
def dinero(s):
 s=limpio(s).replace('$','').replace(' ','')
 if not s:return ''
 if ',' in s and '.' not in s:s=s.replace(',','.')
 try:
  d=Decimal(s)
  return str(d.quantize(Decimal('1'),rounding=ROUND_HALF_UP)) if d.is_finite() else ''
 except InvalidOperation:return ''
def clave_oferta(c,rut,monto,detalle,sel):
 t='|'.join([c.upper(),rut.upper().replace('.','').replace('-',''),monto,detalle.casefold().strip(),sel.casefold().strip()])
 return hashlib.sha256(t.encode('utf-8')).hexdigest()
def poner_proceso(r):
 c=limpio(r['CodigoCotizacion']).upper()
 if not c:return False
 monto=dinero(r.get('MontoTotalDisponble',''))
 fila=[c,limpio(r.get('NombreCotizacion')),limpio(r.get('Estado')),'',
       limpio(r.get('FechaPublicacionParaCotizar')),limpio(r.get('FechaCierreParaCotizar')),
       monto,limpio(r.get('RazonSocialUnidaddeCompra')),limpio(r.get('Region')),
       '','','','','','','Compra Ágil']
 calidad=sum(bool(x) for x in fila[1:9])
 previo=db.execute('SELECT calidad FROM procesos WHERE clave=?',(c,)).fetchone()
 if previo is None or calidad>previo[0]:
  db.execute('INSERT OR REPLACE INTO procesos VALUES (?,?,?)',(c,json.dumps(fila,ensure_ascii=False),calidad))
 return True
def poner_oferta(r):
 c=limpio(r['CodigoCotizacion']).upper()
 prov=limpio(r.get('RazonSocialProveedor'));rut=limpio(r.get('RUTProveedor'))
 monto=dinero(r.get('MontoTotal'));detalle=limpio(r.get('DetalleCotizacion'))
 sel=limpio(r.get('ProveedorSeleccionado'))
 if not(c and prov and rut and monto):return False
 k=clave_oferta(c,rut,monto,detalle,sel)
 fila=[c,prov,monto,'','Histórico ChileCompra Compra Ágil '+zip_name,k,sel]
 db.execute('INSERT OR IGNORE INTO ofertas VALUES (?,?)',(k,json.dumps(fila,ensure_ascii=False)))
 return True
total=0;validas_proceso=0;validas_oferta=0;archivos=[]
with zipfile.ZipFile(io.BytesIO(subidos[zip_name])) as z:
 nombres=[x for x in z.namelist() if x.lower().endswith('.csv')]
 assert len(nombres)==2, f'Se esperaban dos CSV, encontrados: {nombres}'
 for name in nombres:
  inicio=total
  with z.open(name) as binary:
   stream=io.TextIOWrapper(binary,encoding='latin-1',newline='')
   reader=csv.DictReader(stream,delimiter=';')
   oblig={'CodigoCotizacion','NombreCotizacion','MontoTotal','RUTProveedor','RazonSocialProveedor'}
   assert oblig.issubset(set(reader.fieldnames)),f'Columnas faltantes en {name}'
   for r in reader:
    total+=1
    validas_proceso+=int(poner_proceso(r))
    validas_oferta+=int(poner_oferta(r))
    if total%20000==0:
     db.commit()
     if total%200000==0:print('Filas procesadas:',total)
  db.commit();archivos.append({'archivo':name,'filas':total-inicio})
print('Lectura única completada:',archivos)
nproc=db.execute('SELECT COUNT(*) FROM procesos').fetchone()[0]
nof=db.execute('SELECT COUNT(*) FROM ofertas').fetchone()[0]
reporte={'archivos':archivos,'filas_originales':total,'procesos_unicos':nproc,'ofertas_unicas_derivadas':nof,
         'filas_validas_proceso':validas_proceso,'filas_validas_oferta':validas_oferta,
         'ofertas_repetidas_o_indistinguibles':validas_oferta-nof,
         'clave_oferta':'SHA256(codigo,RUT,monto,detalle,seleccionado), no oficial'}
print(json.dumps(reporte,ensure_ascii=False,indent=2))
db.close()
marca=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
respaldo=drive.files().create(body={'name':'CompraAgil_preparado_'+marca+'.sqlite'},
 media_body=MediaFileUpload(dbpath,mimetype='application/octet-stream',resumable=True),
 fields='id,name,webViewLink').execute()
PREPARADO_ID=respaldo['id']
print('PREPARADO_ID:',PREPARADO_ID,'tamaño SQLite MB:',round(os.path.getsize(dbpath)/1e6,1))

## Recuperación opcional sin ZIP

Solo si cambiaste de sesión o cerraste Colab: pega el `PREPARADO_ID` impreso anteriormente. No vuelvas a subir el histórico.

In [ ]:
RECUPERAR_ID=''
if RECUPERAR_ID:
 buf=io.FileIO(dbpath,'wb')
 downloader=MediaIoBaseDownload(buf,drive.files().get_media(fileId=RECUPERAR_ID))
 fin=False
 while not fin:_,fin=downloader.next_chunk()
 buf.close()
 assert os.path.getsize(dbpath)>0
 print('SQLite restaurado:',round(os.path.getsize(dbpath)/1e6,1),'MB')

## Verificación y escritura controlada

Los dos destinos se cargan en la misma sesión sin releer ZIP. Configura `CARGAR='NINGUNA'` para simular, o `'AMBAS'`, `'Procesos'`, `'Ofertas'`. Se valida la estructura antes de escribir y se respalda el clon. La autorización debe ser explícita. Se amplía la capacidad de cada hoja cuando corresponde. La ejecución no debe correr simultáneamente en dos sesiones.

In [ ]:
CARGAR='NINGUNA' # NINGUNA | AMBAS | Procesos | Ofertas
CONFIRMACION='' # AUTORIZO CARGA EN CLON
db=sqlite3.connect(dbpath)
assert db.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
def claves_existentes(ws,columna):
 vals=[v.strip() for v in ws.col_values(columna)[1:] if v.strip()]
 assert len(vals)==len(set(vals)),'Duplicados en destino: '+ws.title
 return set(vals)
def ejecutar(tabla,headers,col,destino):
 assert destino.row_values(1)==headers
 existentes=claves_existentes(destino,col)
 n=db.execute('SELECT COUNT(*) FROM '+tabla).fetchone()[0]
 pendientes=n-sum(1 for (k,) in db.execute('SELECT clave FROM '+tabla) if k in existentes)
 print(tabla,'en SQLite:',n,'existentes:',n-pendientes,'pendientes:',pendientes)
 if CARGAR=='NINGUNA':return
 # Para grandes volúmenes recorrer la base local en bloques, no cargar todo en RAM.
 nuevos=[(k,json.loads(f)) for k,f in db.execute('SELECT clave,fila FROM '+tabla) if k not in existentes]
 requerido=len(existentes)+len(nuevos)+1
 if destino.row_count<requerido:
  destino.add_rows(requerido-destino.row_count)
 escritos=0
 for inicio in range(0,len(nuevos),250):
  lote=nuevos[inicio:inicio+250]
  pos=len(existentes)+2
  fin=pos+len(lote)-1
  rango=f'A{pos}:{chr(64+len(headers))}{fin}'
  destino.update(range_name=rango,values=[fila for _,fila in lote],value_input_option='RAW')
  confirmadas=destino.get(rango)
  idx=headers.index('codigo' if tabla=='procesos' else 'id_cotizacion')
  assert {str(v[idx]) for v in confirmadas}=={k for k,_ in lote},'Lote sin confirmar'
  existentes.update(k for k,_ in lote)
  escritos+=len(lote)
  if escritos%5000==0 or escritos==len(nuevos):print(tabla,'insertados:',escritos)
  time.sleep(0.5)
 print(tabla,'TERMINADA; total:',len(existentes))
assert CARGAR in ('NINGUNA','AMBAS','Procesos','Ofertas')
if CARGAR!='NINGUNA':
 assert CONFIRMACION=='AUTORIZO CARGA EN CLON'
 assert CLON=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
 copia=drive.files().copy(fileId=CLON,body={'name':'Respaldo Compra Agil clon '+datetime.datetime.now().isoformat()},fields='id,webViewLink').execute()
 print('RESPALDO:',copia.get('webViewLink',copia['id']))
if CARGAR in ('NINGUNA','AMBAS','Procesos'):ejecutar('procesos',HEAD_P,1,sp)
if CARGAR in ('NINGUNA','AMBAS','Ofertas'):ejecutar('ofertas',HEAD_O,6,so)
print('Terminado: sin modificación de la base original')